# 🔍 WEEK 9 — Chains, AgentExecutor, and LangGraph

## 🛠️ 0. Setup

In [ ]:
# HINT: %pip install -q these packages: langchain, langchain-classic, langchain-openai, langgraph, python-dotenv
# HINT: langchain-classic is needed because LangChain 1.0 moved AgentExecutor out of langchain.agents


In [ ]:
# HINT: import os, and load_dotenv / find_dotenv from dotenv
# HINT: load the .env file with override=True
# HINT: read OPENAI_API_KEY into api_key and print whether it loaded (show only the first few characters)


## 1️⃣ Part A — LangChain Chains (LCEL)

In [ ]:
# HINT: from langchain_core.prompts import PromptTemplate
# HINT: from langchain_core.output_parsers import StrOutputParser
# HINT: from langchain_openai import ChatOpenAI
# HINT: llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# HINT: build summarize_chain = PromptTemplate.from_template(...) | llm | StrOutputParser()
# HINT: invoke it with a sample text and print the result


### Where Chains Break Down

In [ ]:
# HINT: from langchain_core.runnables import RunnableBranch
# HINT: write is_math_question(inputs) -> True if any character in inputs["question"] is a digit
# HINT: build math_chain and weather_chain (two PromptTemplate | llm | StrOutputParser pipelines, different prompts)
# HINT: routed_chain = RunnableBranch((is_math_question, math_chain), weather_chain)
# HINT: try it with a math question and a weather question — see what happens with the weather one!


## 2️⃣ Part B — AgentExecutor

In [ ]:
# HINT: from langchain_core.tools import tool
# HINT: write a @tool calculator(expression) -> evaluate the expression safely (only allow digits/operators)
# HINT: write a @tool get_weather(location) -> look up a mock dictionary of cities
# HINT: tools = [calculator, get_weather]
# HINT: print the tool names


In [ ]:
# HINT: from langchain_classic.agents import AgentExecutor, create_tool_calling_agent  (NOT langchain.agents!)
# HINT: from langchain_core.prompts import ChatPromptTemplate
# HINT: build agent_prompt with a system message, a {input} human message, and ("placeholder", "{agent_scratchpad}")
# HINT: agent = create_tool_calling_agent(llm, tools, agent_prompt)
# HINT: agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


In [ ]:
# HINT: call agent_executor.invoke({"input": "..."}) with a question that needs BOTH tools
# HINT: print response["output"]


### Where AgentExecutor Breaks Down

In [ ]:
# HINT: build a second AgentExecutor with return_intermediate_steps=True
# HINT: invoke it, then loop over result["intermediate_steps"] (each item is an (action, observation) pair)
# HINT: print action.tool, action.tool_input, and observation for each step
# HINT: print the final result["output"]


In [ ]:
# HINT: build a third AgentExecutor with max_iterations=1
# HINT: invoke it with the same multi-tool question and print the output — what happens?


## 3️⃣ Part C — The Same Agent, Rebuilt in LangGraph

```
        ┌────────┐
        │ agent  │
        └───┬────┘
            │ tool_calls present?
      ┌─────┴─────┐
     yes          no
      │            │
      ▼            ▼
 ┌────────┐       END
 │ tools  │
 └────┬───┘
      │
      └──────► back to agent
```

In [ ]:
# HINT: from typing import Annotated, List, TypedDict
# HINT: from langchain_core.messages import BaseMessage, HumanMessage
# HINT: from langgraph.graph.message import add_messages
# HINT: class AgentState(TypedDict): messages: Annotated[List[BaseMessage], add_messages]
# HINT: llm_with_tools = llm.bind_tools(tools)
# HINT: def agent_node(state): call llm_with_tools.invoke(state["messages"]) and return {"messages": [response]}
# HINT: def should_continue(state): look at state["messages"][-1].tool_calls -> return "tools" or END


In [ ]:
# HINT: from langgraph.graph import StateGraph, START, END
# HINT: from langgraph.prebuilt import ToolNode
# HINT: builder = StateGraph(AgentState); add "agent" node and "tools" node (ToolNode(tools))
# HINT: builder.add_edge(START, "agent")
# HINT: builder.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
# HINT: builder.add_edge("tools", "agent")
# HINT: agent_graph = builder.compile()
# HINT: wrap agent_graph.get_graph().draw_mermaid() in a try/except and print it


In [ ]:
# HINT: agent_graph.invoke({"messages": [HumanMessage(content="...")]})
# HINT: print result["messages"][-1].content
# HINT: compare this answer to the AgentExecutor answer from Part B — it should match!


### Debugging the LangGraph Version

In [ ]:
# HINT: for step in agent_graph.stream({"messages": [HumanMessage(content="...")]}):
# HINT:     get the node name (the single key in step)
# HINT:     loop through step[node_name]["messages"] and print each message's type + content/tool_calls


In [ ]:
# HINT: from langgraph.errors import GraphRecursionError
# HINT: wrap agent_graph.invoke(..., {"recursion_limit": 1}) in a try/except
# HINT: catch GraphRecursionError and print a clean message
# HINT: compare this to the max_iterations cutoff from Part B — same safety idea, much cleaner failure


## 🎯 Try It Yourself

Add a third tool (e.g., a `word_counter` tool, same as Week 4) to the `tools` list and re-run the LangGraph version — notice you don't touch `should_continue`, the conditional edge, or the graph structure at all. Now try the same addition in the `RunnableBranch` chain from Part A and see how much you have to rewrite.

In [ ]:
# HINT: Add a word_counter @tool to the tools list (same shape as calculator/get_weather)
# HINT: Re-run the LangGraph version with the new tool — should_continue and the graph never need to change
# HINT: (Bonus) Try adding the same tool to the RunnableBranch chain from Part A and compare the effort


## 4️⃣ Comparison Table

## 5️⃣ Why LangGraph Exists — Tying It Back

## ✅ Key Takeaways